In [1]:
from pathlib import Path
import sys

# common/ lives in BuildingWithClaudeAPI/, a sibling of this folder.
# Walk up from the current directory until BuildingWithClaudeAPI/common is found.
sys.path.append(r"C:\Users\SValathalaKu\Data\claude-bedrock-learning\BuildingWithClaudeAPI")
from common.aws_config import aws_env_setup
#from common.bedrock_config import get_anthropic_client

aws_env_setup()
#client, model = get_anthropic_client()


In [2]:
from anthropic import AnthropicBedrock

def get_anthropic_client():

     client = AnthropicBedrock(
      aws_region="ap-southeast-2"
      )
     model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"
     return client, model
client, model = get_anthropic_client()

In [6]:
import json
from pathlib import Path

from anthropic import beta_tool


@beta_tool
def read_text_file(filename: str) -> str:
    """Read the complete contents of a UTF-8 text file.

    Args:
        filename: Name of the .txt file in the current directory.
    """
    path = Path(filename)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {filename}")

    if path.suffix.lower() != ".txt":
        raise ValueError("Only .txt files are supported")

    return path.read_text(encoding="utf-8")


@beta_tool
def count_file_words(filename: str) -> str:
    """Count the words in a UTF-8 text file.

    Args:
        filename: Name of the .txt file to count.
    """
    content = read_text_file(filename)

    return json.dumps({
        "filename": filename,
        "word_count": len(content.split())
    })


@beta_tool
def search_text_file(filename: str, search_term: str) -> str:
    """Find lines containing a word or phrase in a text file.

    Args:
        filename: Name of the .txt file to search.
        search_term: Word or phrase to find.
    """
    if not search_term.strip():
        raise ValueError("Search term cannot be empty")

    content = read_text_file(filename)

    matches = [
        {"line_number": number, "text": line}
        for number, line in enumerate(content.splitlines(), start=1)
        if search_term.lower() in line.lower()
    ]

    return json.dumps({
        "filename": filename,
        "search_term": search_term,
        "matches": matches
    })


final_message = client.beta.messages.tool_runner(
    model=model,
    max_tokens=1000,
    tools=[
        read_text_file,
        count_file_words,
        search_text_file
    ],
    messages=[
        {
            "role": "user",
            "content": (
                "Read data3.txt, count its words, find every line "
                "containing 'tool', and provide a concise summary."
            )
        }
    ],
).until_done()


for block in final_message.content:
    if block.type == "text":
        print(block.text)


## Summary

**File:** data3.txt  
**Word Count:** 122 words

**Lines containing 'tool':** 11 lines found

1. Line 1: "Yes. run_tool() can call any function registered in the tool_functions dictionary."
2. Line 4: "tool_functions = {"
3. Line 12: "tool_name comes from Claude's tool_use block."
4. Line 13: ".get(tool_name) retrieves the matching Python function."
5. Line 14: "function(**tool_input) executes that function using Claude's input as named arguments."
6. Line 19: "tool_name = "search_text_file""
7. Line 21: "tool_input = {"
8. Line 34: "The function must be registered in tool_functions."
9. Line 35: "The dictionary key must match the tool schema's name."
10. Line 36: "The keys in tool_input must match the function's parameter names."
11. Line 39: "Memory: Tool schema tells Claude what exists; dispatcher maps the name; Python function performs the work."

**Content Overview:** The file is a technical explanation of how tool functions work in Python. It describes a dispatcher me